# How to Work with Grouped Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ing-bank/probatus/blob/master/docs/howto/grouped_data.ipynb)

Data science often involves naturally grouped data, such as multiple samples from the same customer. In these cases, it is crucial to ensure that all samples from a given group remain in the same fold during cross-validation.

Let's begin by preparing a dataset that includes groups.

In [ ]:
%%capture
!pip install probatus

In [1]:
from sklearn.datasets import make_classification

# Generate a synthetic dataset with 100 samples and 10 features.
X, y = make_classification(n_samples=100, n_features=10, random_state=42)

# Create group labels that cycle through 0 to 4.
groups = [i % 5 for i in range(100)]

# Display the first 10 group labels.
print(groups[:10])

[0, 1, 2, 3, 4, 0, 1, 2, 3, 4]


The integers in the `groups` variable represent the group ID for each sample.

A straightforward way to ensure that data is split based on these groups is to use `GroupKFold` from `sklearn.model_selection`. For additional methods to handle grouped data splits, refer to the [scikit-learn documentation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data).

In [2]:
from sklearn.model_selection import GroupKFold

# Create a GroupKFold object with 5 splits and generate cross-validation indices
cv = list(GroupKFold(n_splits=5).split(X, y, groups=groups))

This variable can be passed to the `cv` parameter in probatus and to hyperparameter optimization tools, such as `RandomizedSearchCV`.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from probatus.features import ShapRFECV

# Initialize the RandomForestClassifier with a fixed random state.
model = RandomForestClassifier(random_state=42)

# Define the hyperparameter grid.
param_grid = {
    "n_estimators": [5, 7, 10],
    "max_leaf_nodes": [3, 5, 7, 10],
}

# Set up RandomizedSearchCV using the defined parameter grid and group-based cross-validation.
search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, cv=cv, n_iter=1, random_state=42)

# Initialize ShapRFECV with the RandomizedSearchCV model and specified parameters.
shap_elimination = ShapRFECV(model=search, step=0.2, cv=cv, scoring="roc_auc", n_jobs=3, random_state=42)

# Execute feature elimination and compute the SHAP report.
report = shap_elimination.fit_compute(X, y)

Feature Elimination:   0%|          | 0/9 [00:00<?, ?it/s]

In [4]:
report

,num_features,features_set,eliminated_features,train_metric_mean,train_metric_std,val_metric_mean,val_metric_std
1,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[6, 7]",0.999562,0.000876,0.954945,0.090110
2,8,"[0, 1, 2, 3, 4, 5, 8, 9]",[5],0.999118,0.001081,0.945513,0.089606
3,7,"[0, 1, 2, 3, 4, 8, 9]",[4],0.999559,0.000548,0.928749,0.137507
4,6,"[0, 1, 2, 3, 8, 9]",[8],0.999179,0.001051,0.969288,0.058854
5,5,"[0, 1, 2, 3, 9]",[9],0.999748,0.000237,0.961767,0.066540
6,4,"[0, 1, 2, 3]",[1],0.999433,0.000700,0.950816,0.090982
7,3,"[0, 2, 3]",[0],0.999120,0.000729,0.970596,0.051567
8,2,"[2, 3]",[3],0.999496,0.000617,0.938639,0.117736
9,1,[2],[],0.998424,0.001819,0.938339,0.097936
